In [2]:
pip install pandera pandas pyarrow fastparquet

Note: you may need to restart the kernel to use updated packages.


In [21]:
import pandas as pd
import pandera as pa
from pandera import Field, DataFrameModel
import glob
import re
from datetime import datetime
from typing import Optional, Union
import numpy as np

RENOVATION_MAPPING = {

    'before clean': 'без отделки',
    'clean': 'чистовая отделка',
    'cosmetic done': 'косметический ремонт',
    'designer renovation': 'дизайнерский ремонт',
    'euro': 'евроремонт',
    'good': 'хороший ремонт',
    'needs renovation': 'требуется ремонт',
    'no renovation': 'без отделки',
    'partial renovation': 'частичный ремонт',
    'prime renovation': 'премиум ремонт',
    'renovated': 'сделан ремонт',
    'turnkey': 'под ключ',
    'unknown': None,
    'white box': 'white box',
    

    'без отделки': 'без отделки',
    'чистовая отделка': 'чистовая отделка',
    'косметический ремонт': 'косметический ремонт',
    'дизайнерский ремонт': 'дизайнерский ремонт',
    'евроремонт': 'евроремонт',
    'под ключ': 'под ключ',
    'хороший ремонт': 'хороший ремонт',
    'требуется ремонт': 'требуется ремонт',
    'частичный ремонт': 'частичный ремонт',
    'премиум ремонт': 'премиум ремонт',
    'сделан ремонт': 'сделан ремонт',
    
    'cosmetic': 'косметический ремонт',
    'designer': 'дизайнерский ремонт',
    'premium': 'премиум ремонт',
    'turn key': 'под ключ',
}

BATHROOM_MAPPING = {
    'bathroom_type_matched': 'совмещенный',
    'bathroom_type_separate': 'раздельный',
    'bathroom_type_multiple': 'раздельный (3 и более)',
    'bathroom_type_separated': 'раздельный',
    'bathroom_type_two_and_more': 'раздельный (2 и более)',
    'bathroom_type_combined': 'совмещенный',
    'совмещенный': 'совмещенный',
    'раздельный': 'раздельный',
    'раздельный (2 и более)': 'раздельный (2 и более)',
    'раздельный (3 и более)': 'раздельный (3 и более)',
}

HOUSE_TYPE_MAPPING = {
    'house_type_monolithic': 'монолитный',
    'house_type_brick': 'кирпичный',
    'house_type_panel': 'панельный',
    'house_type_block': 'блочный',
    'house_type_wooden': 'деревянный',
    'house_type_monolithic_brick': 'монолитно-кирпичный',
    'house_type_monolith': 'монолитный',
    'монолитный': 'монолитный',
    'кирпичный': 'кирпичный',
    'панельный': 'панельный',
    'блочный': 'блочный',
    'деревянный': 'деревянный',
    'монолитно-кирпичный': 'монолитно-кирпичный',
}

ROOMS_MAPPING = {
    'studio': 'студия',
    'студия': 'студия',
}

def normalize_renovation(value: Union[str, float, None]) -> Optional[str]:

    if pd.isna(value) or value == '' or value == 'nan':
        return None
    value = str(value).strip().lower()
    if value in RENOVATION_MAPPING:
        return RENOVATION_MAPPING[value]

    for key, mapped in RENOVATION_MAPPING.items():
        if key in value or value in key:
            return mapped
    return None

def normalize_bathroom(value: Union[str, float, None]) -> Optional[str]:

    if pd.isna(value) or value == '' or value == 'nan':
        return None
    value = str(value).strip().lower()
    if value in BATHROOM_MAPPING:
        return BATHROOM_MAPPING[value]
    return value

def normalize_house_type(value: Union[str, float, None]) -> Optional[str]:

    if pd.isna(value) or value == '' or value == 'nan':
        return None
    value = str(value).strip().lower()
    if value in HOUSE_TYPE_MAPPING:
        return HOUSE_TYPE_MAPPING[value]
    return value

def normalize_rooms(value: Union[str, float, None]) -> Union[int, str, None]:

    if pd.isna(value) or value == '' or value == 'nan':
        return None
    
    value = str(value).strip().lower()
    
    if value in ROOMS_MAPPING or 'студи' in value:
        return 'студия'
    
    try:
        num = int(float(value))
        if 1 <= num <= 10:
            return num
    except (ValueError, TypeError):
        pass
    
    return None


class RealtySchema(DataFrameModel):

    offer_id: str = Field(nullable=False, unique=True)
    url: str = Field(nullable=False)
    
    price: int = Field(
        nullable=True,
        ge=500_000,
        le=5_000_000_000
    )
    
    total_area: float = Field(nullable=True, ge=9, le=1000)
    area_living: float = Field(nullable=True, ge=0)
    area_kitchen: float = Field(nullable=True, ge=0)
    
    rooms: str = Field(nullable=True)
    floor: float = Field(nullable=True)
    total_floors: float = Field(nullable=True)
    
    address: str = Field(nullable=True)
    description: str = Field(nullable=True)
    
    ceiling_height: float = Field(nullable=True, ge=2, le=10)
    build_year: float = Field(nullable=True, ge=1900, le=datetime.now().year + 5)
    
    house_type: str = Field(nullable=True)
    bathroom: str = Field(nullable=True)
    renovation: str = Field(nullable=True)
    
    title: str = Field(nullable=True)
    images_count: float = Field(nullable=True, ge=0)
    s3_uris_count: float = Field(nullable=True, ge=0)
    s3_uris: str = Field(nullable=True)
    
    class Config:
        strict = False
        coerce = True
        name = "realty_data_schema"



def clean_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:

    numeric_cols = ['price', 'total_area', 'area_living', 'area_kitchen', 
                    'floor', 'total_floors', 'ceiling_height', 'build_year',
                    'images_count', 's3_uris_count']
    
    for col in numeric_cols:
        if col in df.columns:

            df[col] = df[col].astype(str).str.replace(',', '.')
            # Конвертируем в numeric
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    if 'ceiling_height' in df.columns:
        df.loc[df['ceiling_height'] > 10, 'ceiling_height'] = pd.NA
        df.loc[df['ceiling_height'] < 1, 'ceiling_height'] = pd.NA
    
    if 'total_area' in df.columns:
        df.loc[df['total_area'] < 8, 'total_area'] = pd.NA
    
    return df


def clean_categorical_columns(df: pd.DataFrame) -> pd.DataFrame:

    if 'renovation' in df.columns:
        df['renovation'] = df['renovation'].apply(normalize_renovation)
        print(f"   Нормализовано renovation: уникальных значений {df['renovation'].nunique()}")
        print(f"      Значения: {df['renovation'].dropna().unique().tolist()}")
    
    if 'bathroom' in df.columns:
        df['bathroom'] = df['bathroom'].apply(normalize_bathroom)
    
    if 'house_type' in df.columns:
        df['house_type'] = df['house_type'].apply(normalize_house_type)
    
    if 'rooms' in df.columns:
        df['rooms'] = df['rooms'].apply(normalize_rooms)
        # Преобразуем в строку для совместимости с parquet
        df['rooms'] = df['rooms'].astype(str)
        df.loc[df['rooms'] == 'nan', 'rooms'] = pd.NA
    
    return df


def remove_anomalies(df: pd.DataFrame) -> pd.DataFrame:

    initial_count = len(df)
    
    df = df.dropna(subset=['price'])
    df = df[df['price'] > 0]

    df = df.dropna(subset=['total_area'])
    df = df[df['total_area'] > 0]
    
    df = df[df['address'].notna()]
    
    if 'total_area' in df.columns:
        df['_temp_price_per_m2'] = df['price'] / df['total_area']
        df = df[(df['_temp_price_per_m2'] >= 30000) & (df['_temp_price_per_m2'] <= 3000000)]
        df = df.drop(columns=['_temp_price_per_m2'])
    

    if 'floor' in df.columns:
        df = df[df['floor'] <= 150]
    if 'total_floors' in df.columns:
        df = df[df['total_floors'] <= 150]
    
    current_year = datetime.now().year
    if 'build_year' in df.columns:
        df = df[(df['build_year'] >= 1900) & (df['build_year'] <= current_year + 5)]
    
    removed = initial_count - len(df)
    print(f"   Удалено аномалий: {removed} записей")
    
    return df


def load_and_merge_csvs(directory: str = ".") -> pd.DataFrame:

    csv_files = glob.glob(f"{directory}/realty_data_*.csv")
    csv_files.extend(glob.glob(f"{directory}/*realty*.csv"))
    csv_files = list(set(csv_files))
    
    if not csv_files:
        raise FileNotFoundError("CSV файлы не найдены")
    
    print(f" Найдено файлов: {len(csv_files)}")
    
    all_dfs = []
    for file in sorted(csv_files):
        try:
            df = pd.read_csv(file, delimiter=';', encoding='utf-8-sig', dtype={'offer_id': str})

            if 'timestamp' in df.columns:
                df = df.drop(columns=['timestamp'])
            all_dfs.append(df)
            print(f"    {file}: {len(df)} записей")
        except Exception as e:
            print(f"    Ошибка {file}: {e}")
    
    combined = pd.concat(all_dfs, ignore_index=True)
    initial_count = len(combined)
    
    combined = combined.drop_duplicates(subset=['offer_id'], keep='last')
    print(f" Уникальных offer_id: {len(combined)} (удалено {initial_count - len(combined)} дубликатов)")

    return combined


def add_additional_checks(df: pd.DataFrame) -> pd.DataFrame:

    initial_count = len(df)
    issues = []
    
    if 'area_living' in df.columns and 'total_area' in df.columns:
        mask = (df['area_living'] > df['total_area']) & (df['area_living'].notna()) & (df['total_area'].notna())
        if mask.any():
            issues.append(f"жилая площадь > общей ({mask.sum()} записей)")
            df = df[~mask]

    if 'area_kitchen' in df.columns and 'total_area' in df.columns:
        mask = (df['area_kitchen'] > df['total_area']) & (df['area_kitchen'].notna()) & (df['total_area'].notna())
        if mask.any():
            issues.append(f"площадь кухни > общей ({mask.sum()} записей)")
            df = df[~mask]
    
    if 'floor' in df.columns and 'total_floors' in df.columns:
        mask = (df['floor'] > df['total_floors']) & (df['floor'].notna()) & (df['total_floors'].notna())
        if mask.any():
            issues.append(f"этаж > всего этажей ({mask.sum()} записей)")
            df = df[~mask]
    
    if issues:
        print(f"    Удалено записей с проблемами:")
        for issue in issues:
            print(f"      - {issue}")
    
    removed = initial_count - len(df)
    if removed > 0:
        print(f"   Всего удалено: {removed} записей")
    
    return df


def add_ml_features(df: pd.DataFrame) -> pd.DataFrame:

    if 'price' in df.columns and 'total_area' in df.columns:
        df['price_per_m2'] = df['price'] / df['total_area']

    if 'floor' in df.columns and 'total_floors' in df.columns:
        def categorize_floor(row):
            if pd.isna(row['floor']) or pd.isna(row['total_floors']) or row['total_floors'] == 0:
                return 'unknown'
            ratio = row['floor'] / row['total_floors']
            if ratio <= 0.33:
                return 'low'
            elif ratio <= 0.66:
                return 'middle'
            else:
                return 'high'
        df['floor_category'] = df.apply(categorize_floor, axis=1)
    
    if 'rooms' in df.columns and 'total_area' in df.columns:
        def classify_apartment(row):
            rooms_val = str(row['rooms']) if pd.notna(row['rooms']) else ''
            if 'студия' in rooms_val:
                return 'studio'
            elif pd.isna(row['rooms']):
                return 'unknown'
            else:
                try:
                    num_rooms = int(float(rooms_val))
                    if num_rooms <= 2:
                        return 'small'
                    elif num_rooms <= 4:
                        return 'medium'
                    else:
                        return 'large'
                except:
                    return 'unknown'
        df['apartment_type'] = df.apply(classify_apartment, axis=1)
    
    if 'description' in df.columns:
        desc = df['description'].fillna('').astype(str)
        df['desc_length'] = desc.str.len()
        df['has_renovation'] = desc.str.contains('ремонт|отделка', case=False, na=False).astype(int)
        df['has_parking'] = desc.str.contains('паркинг|гараж', case=False, na=False).astype(int)
        df['has_balcony'] = desc.str.contains('балкон|лоджия', case=False, na=False).astype(int)
    
    return df


def convert_to_parquet_compatible(df: pd.DataFrame) -> pd.DataFrame:

    for col in df.columns:
        if df[col].dtype == 'Int64' or df[col].dtype == 'int64':
            if df[col].isna().any():
                df[col] = df[col].astype(float)
    
    return df


def process_pipeline(input_dir: str = ".", output_file: str = None) -> pd.DataFrame:

    print("="*70)
    print(" ЗАПУСК ПАЙПЛАЙНА ОБРАБОТКИ ДАННЫХ")
    print("="*70)

    print("\n 1. ЗАГРУЗКА И ОБЪЕДИНЕНИЕ ДАННЫХ...")
    df = load_and_merge_csvs(input_dir)
    
    if df.empty:
        raise ValueError("Нет данных для обработки")
    
    print(f" Всего записей после объединения: {len(df)}")
    
    print("\n 2. ПРИВЕДЕНИЕ ТИПОВ ДАННЫХ...")
    df = clean_numeric_columns(df)
    df = clean_categorical_columns(df)
    
    print("\n 3. УДАЛЕНИЕ АНОМАЛИЙ...")
    df = remove_anomalies(df)
    
    print("\n 4. ДОПОЛНИТЕЛЬНЫЕ ПРОВЕРКИ...")
    df = add_additional_checks(df)
    
    print("\n 5. ВАЛИДАЦИЯ ЧЕРЕЗ PANDERA...")
    try:
        validated_df = RealtySchema.validate(df, lazy=True)
        print(f"    Валидация успешна! {len(validated_df)} записей прошло проверку")
    except pa.errors.SchemaErrors as err:
        error_count = len(err.failure_cases) if hasattr(err, 'failure_cases') else 0
        print(f"    Найдено {error_count} ошибок валидации")
        
        if hasattr(err, 'failure_cases') and len(err.failure_cases) > 0:
            print("\n   Примеры ошибок:")
            failure_cols = err.failure_cases['column'].value_counts().head(5)
            for col, count in failure_cols.items():
                print(f"      - {col}: {count} ошибок")
        
        validated_df = err.data if hasattr(err, 'data') else df
        print(f"    Оставлено валидных записей: {len(validated_df)}")
    
    print("\n 6. ДОБАВЛЕНИЕ ПРИЗНАКОВ ДЛЯ ML...")
    validated_df = add_ml_features(validated_df)
    
    print("\n 7. ПОДГОТОВКА К СОХРАНЕНИЮ...")
    validated_df = convert_to_parquet_compatible(validated_df)
    
    if output_file is None:
        output_file = f"realty_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}.parquet"
    
    print(f"\n 8. СОХРАНЕНИЕ В PARQUET...")
    validated_df.to_parquet(output_file, index=False)
    print(f"    Сохранено: {output_file}")
    print(f"    Размер: {len(validated_df)} записей, {len(validated_df.columns)} колонок")
    
    import os
    file_size_mb = os.path.getsize(output_file) / 1024 / 1024
    print(f"    Размер файла: {file_size_mb:.2f} MB")
    
    print("\n 9. ИТОГОВАЯ СТАТИСТИКА:")
    print(f"   Цена: от {validated_df['price'].min():,.0f} до {validated_df['price'].max():,.0f} ₽")
    print(f"   Площадь: от {validated_df['total_area'].min():.1f} до {validated_df['total_area'].max():.1f} м²")
    
    print(f"\n   Распределение типов ремонта:")
    if 'renovation' in validated_df.columns:
        renovation_counts = validated_df['renovation'].value_counts()
        for r_type, count in renovation_counts.head(10).items():
            print(f"      {r_type}: {count} ({count/len(validated_df)*100:.1f}%)")
    
    print("\n" + "="*70)
    print(" ПАЙПЛАЙН УСПЕШНО ЗАВЕРШЁН!")
    print("="*70)
    
    return validated_df

if __name__ == "__main__":
    
    clean_data = process_pipeline(input_dir=".", output_file="realty_clean.parquet")
    
    print("\n ПРИМЕР ОБРАБОТАННЫХ ДАННЫХ (первые 5 строк):")
    display_cols = ['price', 'total_area', 'rooms', 'floor', 'total_floors', 'build_year', 'renovation', 'price_per_m2']
    existing_cols = [col for col in display_cols if col in clean_data.columns]
    if existing_cols:
        print(clean_data[existing_cols].head(5).to_string())
    
    print(f"\n Данные сохранены в realty_clean.parquet")
    print(f"   Загрузить: df = pd.read_parquet('realty_clean.parquet')")
    print(f"   Размер датасета: {len(clean_data)} строк")
    
    csv_output = f"realty_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    clean_data.to_csv(csv_output, sep=';', index=False, encoding='utf-8-sig')
    print(f"\n Также сохранено в CSV: {csv_output}")

 ЗАПУСК ПАЙПЛАЙНА ОБРАБОТКИ ДАННЫХ

 1. ЗАГРУЗКА И ОБЪЕДИНЕНИЕ ДАННЫХ...
 Найдено файлов: 11
    .\realty_data_20260426_185607.csv: 10 записей
    .\realty_data_20260426_214131.csv: 547 записей
    .\realty_data_20260426_233814.csv: 504 записей
    .\realty_data_20260427_033932.csv: 514 записей
    .\realty_data_20260427_213005.csv: 518 записей
    .\realty_data_20260428_002703.csv: 516 записей
    .\realty_data_20260428_021841.csv: 509 записей
    .\realty_data_20260428_121132.csv: 455 записей
    .\realty_data_20260428_141135.csv: 475 записей
    .\realty_data_20260428_160525.csv: 488 записей
    .\realty_data_try2.csv: 46 записей
 Уникальных offer_id: 1726 (удалено 2856 дубликатов)
 Всего записей после объединения: 1726

 2. ПРИВЕДЕНИЕ ТИПОВ ДАННЫХ...
   Нормализовано renovation: уникальных значений 12
      Значения: ['премиум ремонт', 'без отделки', 'косметический ремонт', 'дизайнерский ремонт', 'евроремонт', 'требуется ремонт', 'чистовая отделка', 'частичный ремонт', 'сделан ремо